# 09. Proyek 3: aplikasi prediksi sentimen

Menyimpan paket prediksi, menguji hasil pemuatan, menangani masukan pengguna, dan menjalankan antarmuka Gradio lokal.

**Prasyarat:** modul 05.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Menyiapkan checkpoint mandiri

Notebook ini melatih checkpoint sendiri sehingga tidak bergantung pada keluaran notebook lain. Untuk aplikasi demonstrasi, kita memilih model rata-rata embedding yang ringan. Pemilihan ini didasarkan tujuan pengajaran, bukan klaim model terbaik.

In [2]:
vocab,train,val,test=loaders()
model=MeanClassifier(len(vocab))
fit(model,train,val,epochs=15)
print("Evaluasi:",run_epoch(model,test))
path=ARTIFACTS/"sentiment.pt"
save_mean(model,vocab,path)

Evaluasi: {'loss': 0.42127753297487897, 'accuracy': 0.75, 'macro_f1': 0.7333333492279053, 'confusion': [[24, 24], [0, 48]]}


## 2. Memeriksa paket model

Artefak berisi bobot, kosakata, dimensi, urutan label, panjang maksimum, dan versi tokenizer. Model yang disajikan pengguna harus memakai aturan yang sama dengan pelatihan.

In [3]:
loaded,metadata=load_mean(path)
print({k:v for k,v in metadata.items() if k not in ["state_dict","vocab"]})
ids,lengths,_=next(iter(test))
model.eval()
with torch.inference_mode():
    assert torch.allclose(model(ids,lengths),loaded(ids,lengths))
print("Pemuatan model konsisten.")

{'dim': 32, 'classes': 2, 'max_length': 64, 'tokenizer': 'regex_lower_v1', 'labels': ['negatif', 'positif']}
Pemuatan model konsisten.


## 3. Fungsi prediksi

Softmax menghasilkan distribusi skor kelas. Skor tinggi tidak otomatis menunjukkan prediksi yang andal, terutama pada teks di luar domain latihan. Aplikasi menampilkan bahwa model memakai data sintetis.

In [4]:
for text in ["buku ini baik","pelayanan ini mengecewakan","tidak bagus sama sekali","😊"]:
    print(text,predict(text,loaded,metadata))

buku ini baik {'negatif': 0.40682145953178406, 'positif': 0.5931785106658936}
pelayanan ini mengecewakan {'negatif': 0.5061870217323303, 'positif': 0.4938129782676697}
tidak bagus sama sekali {'negatif': 0.0059462646022439, 'positif': 0.9940537810325623}
😊 {'negatif': 0.32273605465888977, 'positif': 0.6772639155387878}


## 4. Masukan kosong dan teks panjang

Masukan kosong ditolak agar pengguna dapat memperbaikinya. Teks panjang dipotong konsisten dengan metadata. Periksa dampak pemotongan saat mengganti dataset nyata.

In [5]:
try:
    predict("  ",loaded,metadata)
except ValueError as error:
    print(error)
long_text="baik "*200
print("Jumlah token asli:",len(tokenize(long_text)))
print("Batas model:",metadata["max_length"])
print(predict(long_text,loaded,metadata))

Masukkan teks yang tidak kosong
Jumlah token asli: 200
Batas model: 64
{'negatif': 0.3202360272407532, 'positif': 0.6797640323638916}


## 5. Menjalankan antarmuka lokal

1. Buka terminal pada direktori `nlp`.
2. Jalankan `python -m pip install -r requirements-app.txt`.
3. Jalankan `python app.py`.
4. Buka alamat lokal yang tercetak di terminal.

`share=False` mempertahankan peluncuran lokal. Penerbitan aplikasi daring merupakan langkah terpisah. Notebook tidak meluncurkan server saat Run All sehingga eksekusi dapat selesai.

In [6]:
print((ROOT/"app.py").read_text(encoding="utf-8"))

"""Antarmuka lokal. Instal requirements-app.txt, latih model, lalu python app.py."""
from pathlib import Path
import gradio as gr
from nlp_course.engine import load_mean, predict

checkpoint = Path(__file__).parent / 'artifacts/sentiment.pt'
if not checkpoint.exists():
    raise SystemExit('Latih model dahulu: python -m nlp_course.train')
model, metadata = load_mean(checkpoint)

def classify(text):
    if not text.strip():
        raise gr.Error('Masukkan teks terlebih dahulu.')
    return predict(text, model, metadata)

if __name__ == '__main__':
    gr.Interface(fn=classify, inputs=gr.Textbox(label='Ulasan'),
                 outputs=gr.Label(label='Skor sentimen'),
                 title='Latihan klasifikasi sentimen',
                 description='Model pembelajaran dengan data sintetis. Skor belum dikalibrasi.',
                 examples=['layanan ini sangat baik', 'produk ini mengecewakan']).launch(share=False)



## 6. Evaluasi penggunaan dan pengembangan

Sebelum menggunakan model pada ulasan nyata, buat kumpulan evaluasi yang mewakili calon pengguna. Periksa negasi, ejaan tidak baku, campuran bahasa, teks panjang, dan kelas yang tidak seimbang. Tentukan cara memperbarui model dan menyimpan versi dataset.

Jalur berikutnya tersedia pada notebook 10: tokenizer subword, dataset publik, dan fine-tuning BERT kecil. Untuk proyek lanjutan, Anda dapat mengembangkan klasifikasi multikelas, NER, peringkasan, atau tanya jawab dengan protokol evaluasi masing-masing.

## Latihan mandiri

1. Mengapa preprocessing termasuk bagian model?
2. Apa perbedaan aplikasi lokal dan penerbitan daring?
3. Mengapa skor softmax tidak disebut kepastian?
4. Apa yang diuji sebelum mengganti checkpoint aplikasi?

## Pembahasan latihan

1. Bobot menerima indeks numerik yang ditentukan preprocessing. Perubahan aturan mengubah masukan model.
2. Aplikasi lokal berjalan di komputer sendiri. Penerbitan daring membutuhkan lingkungan server serta konfigurasi akses.
3. Softmax dapat memberi skor tinggi pada prediksi yang salah.
4. Konsistensi label, tokenizer, panjang maksimum, pemuatan, dan metrik evaluasi pada data representatif.

## Penghubung ke materi berikutnya

Rangkaian inti selesai. Gunakan notebook 10 untuk berpindah dari eksperimen mekanisme ke data serta model pralatih eksternal.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.